In [ ]:
library(lme4)
library(lmerTest)
library(ggplot2)
library(mgcv)
library(plotly)
library(parallel)
library(MuMIn)
library(ggeffects)
library(car)
library(tidyr)
library(dplyr)
library(glmmTMB)
rescale_01 <- function(x) {
  (x - min(x, na.rm = TRUE)) / (max(x, na.rm = TRUE) - min(x, na.rm = TRUE))
}

residual_test <- function(model){
hist(residuals(model), main = "Histogram of Residuals", xlab = "Residuals")
qqnorm(residuals(model))
qqline(residuals(model), col = "red")
# shapiro.test(residuals(model))
ks.test(residuals(model), "pnorm", mean = mean(residuals(model)), sd = sd(residuals(model)))}

df <- read.csv("../../../data_processing/combine_data/results_aeema.csv")

In [ ]:
df <- df[!df$is_limited, ]
df <- df[!df$cross_syllable_boundary, ]
df <- df[df$data_space=="latent", ]

df[df$modality=="art",]$dist <- scale(df[df$modality=="art",]$dist)
df[df$modality=="ac",]$dist <- scale(df[df$modality=="ac",]$dist)

df$phone1_H_art_given_ac <- scale(df$phone1_H_art_given_ac)
df$phone2_H_art_given_ac <- scale(df$phone2_H_art_given_ac)
df$phone2_freq <- scale(log(df$phone2_freq))

df$subject <- as.factor(df$subject)
df$word <- as.factor(df$word)
df$bigram <- as.factor(paste(df$phone1, df$phone2, sep = "_"))

df$modality <- as.factor(df$modality)
contrasts(df$modality) <- c(-0.5, 0.5)
print(contrasts(df$modality))
print("n rows:")
print(nrow(df))

df$phone2_duration <- df$phone2_end - df$phone2_start
df$phone2_duration <- scale(df$phone2_duration)

In [ ]:
model <- lmer(
  dist ~ phone2_pos_entropy *
         modality *

         phone2_H_art_given_ac *
         phone1_H_art_given_ac *

         phone2_duration *

         phone2_freq +
    (phone2_pos_entropy | word) + 
    (phone2_pos_entropy | bigram) + 
    (phone2_pos_entropy | subject),
  data = df,
  control = lmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 2e5)
  )
)
summary(model)

In [ ]:
library(emmeans)
tr <- emtrends(model, ~ modality, var = "phone2_pos_entropy")
summary(tr, infer = c(TRUE, TRUE)) 

In [ ]:
get_term_base <- function(model, term_name) {
  coef_table <- summary(model)$coefficients
  estimate <- coef_table[term_name, "Estimate"]
  p_value <- if(length(grep("Pr\\(>|p-value", colnames(coef_table), value = TRUE)) > 0) coef_table[term_name, grep("Pr\\(>|p-value", colnames(coef_table), value = TRUE)] else NA
  return(list(sign = ifelse(estimate > 0, "+", "-"), p_value = p_value,significant = (!is.na(p_value) && p_value < 0.05)))
}
get_term_base(model, "phone2_pos_entropy")
get_term_base(model, "phone2_pos_entropy:modality1:phone2_H_art_given_ac")

In [ ]:
qs <- quantile(df$phone2_H_art_given_ac,
               probs = c(0, 0.25, 0.5, 0.75, 1),
               na.rm = TRUE)
vals <- as.numeric(qs)   # convert named quantile vector to numeric


preds <- ggpredict(model, terms = c("phone2_pos_entropy", "modality [ac, art]"))

preds$group <- factor(preds$group,
                      labels = c("ac", "art"))

p <- plot(preds) + labs(color = "Modality") + xlab("Target position entropy (z-scored)") + ylab("Predicted distinctiveness (z-scored)") + ggtitle(NULL) + ylim(-0.75, 1.25)
p
# write.csv(preds, "preds_surprisal.csv", row.names = FALSE)

preds <- ggpredict(model, terms = c("phone2_pos_entropy", sprintf("phone2_H_art_given_ac [%s]",
            paste(round(vals, 4), collapse = ", ")), "modality [ac]"))

preds$group <- factor(preds$group,
                      labels = c("Min", "Q1", "Q2", "Q3", "Max"))

p <- plot(preds) +
  labs(color = "Target phoneme flexibility\n(z-scored)") +
  xlab("Target position entropy (z-scored)") +
  ylab("Predicted acoustic distinctiveness\n(z-scored)") +
  ggtitle(NULL) +
  ylim(-0.75, 1.25) +
  scale_color_viridis_d(option = "viridis") +
  scale_fill_viridis_d(option = "viridis") +
  theme_minimal(base_size = 16)

# ggsave("acousticplot.png", plot = p, width = 8, height = 4.3, dpi = 300)
p
# write.csv(preds, "../preds_acoustics.csv", row.names = FALSE)


preds <- ggpredict(model, terms = c("phone2_pos_entropy", sprintf("phone2_H_art_given_ac [%s]",
            paste(round(vals, 4), collapse = ", ")), "modality [art]"))

preds$group <- factor(preds$group,
                      labels = c("Min", "Q1", "Q2", "Q3", "Max"))

p <- plot(preds) +
  labs(color = "Target phoneme flexibility\n(z-scored)") +
  xlab("Target position entropy (z-scored)") +
  ylab("Predicted articulatory distinctiveness\n(z-scored)") +
  ggtitle(NULL) +
  ylim(-0.75, 1.25) +
  scale_color_viridis_d(option = "viridis") +
  scale_fill_viridis_d(option = "viridis") +
  theme_minimal(base_size = 16)

# ggsave("articulatoryplot.png", plot = p, width = 8, height = 4.3, dpi = 300)
p
# write.csv(preds, "../preds_articulatory.csv", row.names = FALSE)

In [ ]:
library(emmeans)
library(dplyr)
library(ggplot2)

# 1) Slopes (trends) of surprisal across a smooth MSE grid
mse_grid <- seq(min(df$phone2_H_art_given_ac, na.rm = TRUE),
                max(df$phone2_H_art_given_ac, na.rm = TRUE),
                length.out = 75)

em_curve <- emtrends(
  model,
  specs = ~ modality * phone2_H_art_given_ac,
  var   = "phone2_pos_entropy",
  at    = list(phone2_H_art_given_ac = mse_grid)
)

curve_df <- as.data.frame(em_curve) %>%
  # emtrends names the slope column like "surprisal.trend"
  rename(slope = !!grep("trend$", names(.), value = TRUE),
         mse   = phone2_H_art_given_ac)

ggplot(curve_df, aes(x = mse, y = slope, color = modality)) +
  geom_line(size = 1.2) +
  geom_ribbon(aes(ymin = slope - SE, ymax = slope + SE, fill = modality),
              alpha = 0.12, color = NA) +
  labs(x = "Target MSE (z)",
       y = "Slope of surprisal → response",
       color = "Type", fill = "Type",
       title = "How MSE changes the surprisal slope, by modality") +
  theme_minimal(base_size = 14)

diff_curve <- contrast(
  em_curve,
  method = "pairwise",                    # (acoustic - articulatory)
  by     = "phone2_H_art_given_ac"
)

diff_df <- as.data.frame(diff_curve, infer = c(TRUE, TRUE)) %>%
  dplyr::rename(delta_slope = estimate,
                mse         = phone2_H_art_given_ac)

ggplot(diff_df, aes(x = mse, y = delta_slope)) +
  geom_hline(yintercept = 0, linetype = 2) +
  geom_line(size = 1.2) +
  geom_ribbon(aes(ymin = asymp.LCL, ymax = asymp.UCL), alpha = 0.15) +
  labs(x = "Phone2 flexibility (z)",
       y = "Δ slope (acoustic − articulatory)",
       title = "Difference in surprisal slope between modalities across flexibility") +
  theme_minimal(base_size = 14)
# write.csv(diff_df, "../diff_df.csv", row.names = FALSE)